## Preparação

In [1]:
# Conecta o colab ao drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Instala as dependências necessárias
!pip -q install openai pandas pyarrow tqdm python-dotenv

In [36]:
# Importa as bibliotecas necessárias
import os
import getpass
import json
import time
import random
from datetime import datetime, timezone
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
import shutil
import matplotlib.pyplot as plt

## Carrega e configura silver

In [4]:
path_silver = "/content/drive/MyDrive/Colab Notebooks/dadosfera/amazon_products_silver"

df = pd.read_parquet(path_silver, engine="pyarrow")

In [5]:
df.shape

(1379629, 17)

In [6]:
df.columns

Index(['product_id', 'product_title', 'category_id', 'category_name', 'price',
       'list_price', 'rating', 'review_count', 'has_rating', 'weighted_score',
       'units_sold_last_month', 'is_best_seller', 'has_title', 'image_url',
       'product_url', 'price_segment', 'popularity_tier'],
      dtype='object')

### Parâmetros de amostragem

In [7]:
TOP_CATEGORIES = 40
K_PER_GROUP = 3

# Garante colunas
needed = ["product_id", "category_id", "product_title", "price_segment", "popularity_tier"]
missing = [c for c in needed if c not in df.columns]

if missing:
    raise ValueError(f"Colunas faltando na Silver: {missing}")

### Top categorias

In [8]:
top_cats = df["category_id"].value_counts().head(TOP_CATEGORIES).index

df_top = df[df["category_id"].isin(top_cats)].copy()

### Amostra estratificada

In [9]:
sample_rows = []
group_cols = ["category_id", "price_segment", "popularity_tier"]

for _, g in df_top.groupby(group_cols):
    # remove títulos vazios
    g = g[g["product_title"].astype(str).str.len() > 0]
    if len(g) == 0:
        continue
    n = min(K_PER_GROUP, len(g))
    sample_rows.append(g.sample(n=n, random_state=42))

df_sample = pd.concat(sample_rows, ignore_index=True)

### Controle de custos e performance

In [10]:
MAX_SAMPLE = 400

# Se a amostra for maior que o teto, reduz para o limite
if len(df_sample) > MAX_SAMPLE:
    df_sample = df_sample.sample(MAX_SAMPLE, random_state=42).reset_index(drop=True)

In [11]:
df_sample.shape

(400, 17)

In [31]:
# Pré-visualização da estratificação da amostra
df_sample[["category_id", "price_segment", "popularity_tier"]].drop_duplicates().head(10)

,category_id,price_segment,popularity_tier
0,118,luxury,high
1,110,budget,medium
2,105,premium,high
3,16,premium,high
4,43,luxury,high
5,91,budget,high
6,72,luxury,high
7,168,budget,low
8,88,mid,low
9,173,mid,low


## Configurar OpenAI

In [13]:
# Autenticação segura via variável de ambiente
os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

client = OpenAI()

OPENAI_API_KEY: ··········


In [14]:
# Parâmetros globais para controle de geração de texto e fluxo de requisições
MODEL_NAME = "gpt-4o-mini"
MAX_TOKENS = 400
TEMPERATURE = 0
REQUEST_DELAY = 0.8

### Prompt e função de extração

In [15]:
SYSTEM_PROMPT = """Você é um extrator de atributos de catálogo e-commerce.
Dado um título de produto, extraia campos estruturados e responda APENAS em JSON válido.
Não inclua texto fora do JSON.
"""

USER_TEMPLATE = """Extraia os campos abaixo a partir do título:

Campos:
- brand_guess: string ou null
- product_type: string curta
- attributes: array de strings (máx 6)
- keywords: array de strings (3 a 6)
- title_clean: string curta (sem emojis e sem excesso de pontuação)

Regras:
- Responda somente JSON.
- Se não conseguir inferir a marca, use null.
- product_type deve ser uma categoria curta (ex: "headphones", "shampoo", "monitor").

Título: {title}
"""

In [16]:
# Tenta carregar o JSON e, se falhar, tenta extrair o bloco JSON do texto
def safe_json_loads(s: str):
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        start = s.find("{")
        end = s.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(s[start:end+1])
        raise

# Realiza a chamada para a OpenAI com lógica de reativação (retry) e validação de esquema para garantir integridade dos dados
def extract_features(title: str, model: str = "gpt-4o-mini", max_retries: int = 4):
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=model,
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": USER_TEMPLATE.format(title=title)},
                ],
            )
            content = resp.choices[0].message.content
            data = safe_json_loads(content)

            # Valida campos mínimos
            for k in ["brand_guess", "product_type", "attributes", "keywords", "title_clean"]:
                if k not in data:
                    raise ValueError(f"Campo ausente no JSON: {k}")

            # Normaliza tipos
            if data["brand_guess"] == "" or data["brand_guess"] == "null":
                data["brand_guess"] = None
            data["attributes"] = list(data["attributes"])[:6]
            data["keywords"] = list(data["keywords"])[:6]

            return data

        except Exception as e:
            msg = str(e)

            if attempt == max_retries:
                return {"error": msg}

            # Backoff exponencial + jitter (melhor para rate limit)
            sleep_s = min(60, (2 ** attempt)) + random.uniform(0, 1.5)
            time.sleep(sleep_s)

### Execução do processamento

In [17]:
records = []
now = datetime.now(timezone.utc).isoformat()

# Extração de atributos via LLM
for row in tqdm(df_sample.itertuples(index=False), total=len(df_sample)):
    title = str(row.product_title)
    result = extract_features(title, model=MODEL_NAME)
    records.append({
        "product_id": row.product_id,
        "category_id": int(row.category_id),
        "price_segment": str(row.price_segment),
        "popularity_tier": str(row.popularity_tier),
        "product_title": title,
        "llm_brand_guess": result.get("brand_guess"),
        "llm_product_type": result.get("product_type"),
        "llm_attributes_json": json.dumps(result.get("attributes", []), ensure_ascii=False),
        "llm_keywords_json": json.dumps(result.get("keywords", []), ensure_ascii=False),
        "llm_title_clean": result.get("title_clean"),
        "llm_model": MODEL_NAME,
        "created_at_utc": now,
        "llm_error": result.get("error"),
    })

    time.sleep(REQUEST_DELAY)

df_llm = pd.DataFrame(records)

100%|██████████| 400/400 [20:02<00:00,  3.01s/it]


In [18]:
df_llm.head(5)

,product_id,category_id,price_segment,popularity_tier,product_title,llm_brand_guess,llm_product_type,llm_attributes_json,llm_keywords_json,llm_title_clean,llm_model,created_at_utc,llm_error
0,B098KP6D4B,118,luxury,high,Disney Mickey Mouse Mick-O-Lantern Halloween C...,Disney,bag,"[""Mickey Mouse"", ""Halloween"", ""crossbody"", ""pu...","[""Disney"", ""Mickey Mouse"", ""Halloween"", ""cross...",Disney Mickey Mouse Mick-O-Lantern Halloween C...,gpt-4o-mini,2026-03-01T17:26:14.965277+00:00,None
1,B0BXNBBFWX,110,budget,medium,Men's Athletic Polo Shirts Short Sleeve T-Shir...,None,polo shirts,"[""men's"", ""athletic"", ""short sleeve"", ""casual""...","[""men's"", ""athletic"", ""polo"", ""shirts"", ""casual""]",Men's Athletic Polo Shirts Short Sleeve,gpt-4o-mini,2026-03-01T17:26:14.965277+00:00,None
2,B08GN96CMR,105,premium,high,4oz Plastic Containers with Lids 50 Pack BPA F...,None,containers,"[""BPA free"", ""refillable"", ""clear"", ""round"", ""...","[""plastic containers"", ""jars with lids"", ""cosm...",4oz Plastic Containers with Lids 50 Pack,gpt-4o-mini,2026-03-01T17:26:14.965277+00:00,None
3,B016R9KKCU,16,premium,high,JACO ElitePro Tire Pressure Gauge - 100 PSI,JACO,gauge,"[""ElitePro"", ""100 PSI"", ""tire pressure"", ""auto...","[""tire"", ""pressure"", ""gauge"", ""automotive"", ""J...",JACO ElitePro Tire Pressure Gauge,gpt-4o-mini,2026-03-01T17:26:14.965277+00:00,None
4,B09G8RF8V8,43,luxury,high,STAR WARS The Mandalorian The Child 8 Piece La...,STAR WARS,layette set,"[""8 pieces"", ""The Mandalorian"", ""The Child"", ""...","[""STAR WARS"", ""Mandalorian"", ""The Child"", ""lay...",STAR WARS The Mandalorian The Child 8 Piece La...,gpt-4o-mini,2026-03-01T17:26:14.965277+00:00,None


### Checa erros

In [19]:
df_llm["llm_error"].value_counts(dropna=False).head(5)

,count
llm_error,
None,400


In [20]:
df_llm["llm_error"].notna().mean(), df_llm["llm_error"].notna().sum()

(np.float64(0.0), np.int64(0))

### Coleta métricas da execução

In [21]:
total = len(df_llm)
errors = df_llm["llm_error"].notna().sum()
ok = total - errors

print("Total:", total)
print("OK:", ok)
print("Erros:", errors)

Total: 400
OK: 400
Erros: 0


### Define um cenário de tokens e preço

In [22]:
MODEL = "gpt-4o-mini"

tokens_in_avg = 350
tokens_out_avg = 180

price_in = None
price_out = None

In [ ]:
# Valores aproximados para gpt-4o-mini por 1M de tokens
# price_in = 0.15
# price_out = 0.60

### Calcula estimativa

In [23]:
estimated_in_tokens_total = ok * tokens_in_avg
estimated_out_tokens_total = ok * tokens_out_avg

print("Tokens entrada estimados:", estimated_in_tokens_total)
print("Tokens saída estimados:", estimated_out_tokens_total)

if price_in and price_out:
    cost = (estimated_in_tokens_total / 1_000_000) * price_in + (estimated_out_tokens_total / 1_000_000) * price_out
    print("Custo estimado (USD):", cost)
else:
    print("⚠️ Preços não informados — reporte apenas tokens estimados no doc.")

Tokens entrada estimados: 140000
Tokens saída estimados: 72000
⚠️ Preços não informados — reporte apenas tokens estimados no doc.


### Texto para documentação

In [24]:
md = f"""
## 💰 Estimativa de Custo (LLM)

- Modelo: `{MODEL_NAME}`
- Registros processados: {total}
- Sucesso: {ok}
- Falhas: {errors}

### Tokens (estimativa conservadora)
- Tokens médios de entrada por chamada: ~{tokens_in_avg}
- Tokens médios de saída por chamada: ~{tokens_out_avg}
- Tokens totais estimados (entrada): ~{estimated_in_tokens_total:,}
- Tokens totais estimados (saída): ~{estimated_out_tokens_total:,}

> Observação: valores estimados por heurística; podem variar conforme tamanho do título e JSON retornado.
"""
print(md)


## 💰 Estimativa de Custo (LLM)

- Modelo: `gpt-4o-mini`
- Registros processados: 400
- Sucesso: 400
- Falhas: 0

### Tokens (estimativa conservadora)
- Tokens médios de entrada por chamada: ~350
- Tokens médios de saída por chamada: ~180
- Tokens totais estimados (entrada): ~140,000
- Tokens totais estimados (saída): ~72,000

> Observação: valores estimados por heurística; podem variar conforme tamanho do título e JSON retornado.



In [32]:
# Visualiza uma amostra dos resultados limpos
df_llm[["product_title", "llm_product_type", "llm_brand_guess", "llm_error"]].head()

,product_title,llm_product_type,llm_brand_guess,llm_error
0,Disney Mickey Mouse Mick-O-Lantern Halloween C...,bag,Disney,None
1,Men's Athletic Polo Shirts Short Sleeve T-Shir...,polo shirts,None,None
2,4oz Plastic Containers with Lids 50 Pack BPA F...,containers,None,None
3,JACO ElitePro Tire Pressure Gauge - 100 PSI,gauge,JACO,None
4,STAR WARS The Mandalorian The Child 8 Piece La...,layette set,STAR WARS,None


In [35]:
# Visualiza uma contagem por tipo de produto
df_llm["llm_product_type"].value_counts().head()

,count
llm_product_type,
headphones,9
watch,8
decoração,8
backpack,7
toys,7


## Salva camada “enriched” (Parquet)

In [ ]:
path_enriched = "/content/drive/MyDrive/Colab Notebooks/dadosfera/amazon_products_enriched_llm"

# overwrite robusto (arquivo ou pasta)
if os.path.exists(path_enriched):
    if os.path.isdir(path_enriched):
        shutil.rmtree(path_enriched)
    else:
        os.remove(path_enriched)

df_llm.to_parquet(path_enriched, engine="pyarrow", index=False)

In [34]:
print("✅ Saved:", path_enriched)

✅ Saved: /content/drive/MyDrive/Colab Notebooks/dadosfera/amazon_products_enriched_llm


In [41]:
df_llm.shape

(400, 13)

## Join com silver

In [42]:
df_enriched = pd.read_parquet(path_enriched, engine="pyarrow")

df_silver_enriched = df.merge(
    df_enriched.drop(columns=["product_title"]),
    on=["product_id", "category_id", "price_segment", "popularity_tier"],
    how="left"
)

In [43]:
df_silver_enriched.shape

(1379629, 25)

In [44]:
df_silver_enriched.head()

,product_id,product_title,category_id,category_name,price,list_price,rating,review_count,has_rating,weighted_score,...,price_segment,popularity_tier,llm_brand_guess,llm_product_type,llm_attributes_json,llm_keywords_json,llm_title_clean,llm_model,created_at_utc,llm_error
0,B01M3RYHP0,Men's Tag-Free Cotton Briefs,110,Men's Clothing,8.38,22.99,4.6,0,True,0.0,...,budget,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,B07P7M18C6,Energy Unisex Easy-On/Easy-Off Knee High Compr...,110,Men's Clothing,11.88,0.00,4.3,0,True,0.0,...,budget,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,B0B62HGK2H,Hooded Rain Poncho Waterproof Raincoat Jacket ...,110,Men's Clothing,9.99,13.99,4.5,0,True,0.0,...,budget,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,B01L8JJ57U,"Men's EcoSmart Fleece Sweatshirt, Cotton-Blend...",110,Men's Clothing,10.99,18.00,4.6,0,True,0.0,...,budget,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,B08MBFGL13,"Men's Max Cushioned Crew Socks, Moisture-Wicki...",110,Men's Clothing,11.00,0.00,4.6,0,True,0.0,...,budget,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Salva gold enriched

In [45]:
path_gold_dashboard  = "/content/drive/MyDrive/Colab Notebooks/dadosfera/amazon_products_gold_dashboard"

if os.path.exists(path_gold_dashboard):
    if os.path.isdir(path_gold_dashboard):
        shutil.rmtree(path_gold_dashboard)
    else:
        os.remove(path_gold_dashboard)

df_silver_enriched.to_parquet(path_gold_dashboard , engine="pyarrow", index=False)

In [46]:
df_silver_enriched.sample(1000, random_state=42).to_csv(
    "/content/drive/MyDrive/Colab Notebooks/dadosfera/gold_dashboard_sample.csv",
    index=False
)